# 08 Showcase Integration Patterns (OpenClaw, 2026)

## What This Lesson Is
Convert showcase concepts into production-ready integration blueprints with stage contracts.

## Scientific Lens
- Concept: Stage-based decomposition improves debuggability and governance.
- Measure: Stage completion coverage across defined integration pipeline.
- Validity Limit: Showcase patterns must be adapted to real org constraints.


## How It Works
1. Decompose representative workflows into stage contracts.
2. Attach metrics and failure boundaries to each stage.
3. Run live OpenClaw call requesting concrete blueprint and compare to deterministic skeleton.


In [ ]:
import os
from openai import OpenAI

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_client()
    resp = client.chat.completions.create(
        model="openclaw",
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
pipelines = {
    "pr_review_notify": ["collect", "analyze", "decide", "notify"],
    "skill_bootstrap": ["ingest", "draft", "validate", "publish"],
}
critical = {"analyze", "decide", "notify", "validate"}
seen = {s for stages in pipelines.values() for s in stages}
assert critical.issubset(seen)


In [ ]:
# Live Demo
try:
    q = "Design a PR-review-to-Telegram pipeline with stage contracts, metrics, and rollback steps."
    print(ask_openclaw(q, user="showcase-patterns"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add failure injection to one stage and define containment behavior.
2. Define trace schema linking all stages by correlation id.
3. Add policy checks for high-risk stages before execution.

## Validation Checklist
- Pipelines are modeled as stage contracts, not prompt blobs.
- Observability and rollback are first-class in design.
- Live call asks for concrete integration architecture output.

## Further Reading
- OpenClaw showcase: https://docs.openclaw.ai/start/showcase
- OpenClaw overview: https://docs.openclaw.ai/start/openclaw
